In [3]:
import openeo

In [4]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [4]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [112.6604742,-7.1531396
            ],
            [
              112.621704,
              -7.1531396
            ],
            [
              112.621704,
              -7.1855558
            ],
            [
              112.6608208,
              -7.1855558
            ],
            [
              112.6604742,
              -7.1531396
            ]
          ]
        ]
      }
    }
  ]
}

In [1]:
s5_no2 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={"west": 112.621704, "south": -7.1855558, "east": 112.6608208, "north": -7.1531396},    
    bands=["CO"],
)

NameError: name 'connection' is not defined

In [6]:
# Now aggregate by day to avoid having multiple data per day
s5_no2 = s5_no2.aggregate_temporal_period(reducer="mean", period="day")

# let's create a spatial aggregation to generate mean timeseries data
s5_no2 = s5_no2.aggregate_spatial(reducer="mean", geometries=aoi)

In [7]:
job = s5_no2.execute_batch(title="NO2 Gresik", outputfile="polutan_no2_gresik.nc")

0:00:00 Job 'j-2608280754224f72b1aa178f0b3870c1': send 'start'
0:00:02 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:00:08 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:00:14 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:00:22 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:00:32 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:00:45 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:01:01 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:01:20 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:01:44 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:02:14 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:02:52 Job 'j-2608280754224f72b1aa178f0b3870c1': queued (progress 0%)
0:03:38 Job 'j-2608280754224f72b1aa178f0b3870c1': running (progress N/A)
0:04:37 Job 'j-2608280754224f72b1aa178f0b3870c1': running (progress N/A)
0:05:37 Jo

In [7]:
import netCDF4
import numpy as np
import pandas as pd

ds = netCDF4.Dataset("polutan_no2_gresik.nc")

no2 = ds.variables["NO2"][0, :]          # dimensi (feature, t) -> feature 0, semua waktu
time = ds.variables["t"][:]
time_units = ds.variables["t"].units
dates = netCDF4.num2date(time, units=time_units)

# Deret tanggal penuh 1 tahun (24 Agst 2025 - 23 Agst 2026)
full_dates = pd.date_range(start="2025-08-24", end="2026-08-23", freq="D")

no2_map = {d.strftime("%Y-%m-%d"): float(v) for d, v in zip(dates, no2)}
df = pd.DataFrame({"date": full_dates.strftime("%Y-%m-%d")})
df["NO2"] = df["date"].map(no2_map)   # hari tanpa data otomatis jadi NaN

df.to_csv("no2_gresik_timeseries.csv", index=False)
print("CSV disimpan:", len(df), "baris")

CSV disimpan: 365 baris
